# Week 7 - Notebook 3: Fine-tune with QLoRA

## Goal
Fine-tune Llama 3.2 using QLoRA for price prediction:
1. Load base model in 4-bit
2. Configure LoRA adapters
3. Train with SFTTrainer
4. Save adapters to HuggingFace Hub

Expected result: ~$40 error (beats GPT-5.1!)

## Time: 2-12 hours (depending on mode and GPU)

**IMPORTANT:** Run this on Google Colab with GPU!
- Free T4: ~3 hours (Lite mode)
- Paid A100: ~8-12 hours (Full mode)

## Google Colab Setup

1. Upload this notebook to Google Colab
2. Runtime → Change runtime type → GPU (T4 or A100)
3. Run all cells

## Install Dependencies (Colab only)

In [ ]:
# Uncomment and run on Google Colab
# !pip install -q torch transformers datasets peft trl bitsandbytes accelerate python-dotenv huggingface-hub pydantic

## Setup Environment

In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

# Configuration
LITE_MODE = True  # Set to False for full training
HF_TOKEN = "your_token_here"  # Replace with your HuggingFace token

# Login to HuggingFace
login(HF_TOKEN)

print("✅ Environment setup complete")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Configuration

In [ ]:
# Model and dataset
BASE_MODEL = "meta-llama/Llama-3.2-3B"
DATASET_NAME = "ed-donner/items_prompts_lite" if LITE_MODE else "ed-donner/items_prompts_full"
OUTPUT_DIR = "./llama-pricer-lite" if LITE_MODE else "./llama-pricer-full"
HUB_MODEL_ID = "your-username/llama-pricer-lite" if LITE_MODE else "your-username/llama-pricer-full"

# QLoRA settings
LORA_RANK = 16 if LITE_MODE else 32
LORA_ALPHA = 32 if LITE_MODE else 64
LORA_DROPOUT = 0.05

# Training settings
BATCH_SIZE = 4 if LITE_MODE else 8
GRADIENT_ACCUMULATION = 4
NUM_EPOCHS = 3 if LITE_MODE else 5
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 256

print("Configuration:")
print(f"  Mode: {'LITE' if LITE_MODE else 'FULL'}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  LoRA Rank: {LORA_RANK}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")

## Load Dataset

In [ ]:
print(f"Loading dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME)

train_data = dataset["train"]
val_data = dataset["val"]

print(f"✅ Dataset loaded:")
print(f"   Training: {len(train_data):,} examples")
print(f"   Validation: {len(val_data):,} examples")

# Show example
print(f"\nExample training data:")
print(f"Prompt: {train_data[0]['prompt'][:100]}...")
print(f"Completion: {train_data[0]['completion']}")

## Load Base Model with 4-bit Quantization

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {BASE_MODEL}")
print("This may take a few minutes...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # Required for gradient checkpointing

print("✅ Model loaded in 4-bit")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Configure LoRA Adapters

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

# Add LoRA adapters
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = 100 * trainable_params / total_params

print("✅ LoRA adapters configured")
print(f"Trainable parameters: {trainable_params:,} ({trainable_pct:.2f}%)")
print(f"Total parameters: {total_params:,}")

## Configure Training

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    push_to_hub=False,  # We'll push manually later
)

print("✅ Training arguments configured")

## Format Dataset for Training

In [ ]:
def formatting_func(example):
    """Format prompt-completion pairs for training"""
    return example["prompt"] + example["completion"]

print("✅ Formatting function ready")

## Create Trainer

In [ ]:
# Create SFT Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LENGTH,
)

print("✅ Trainer created")
print(f"\nReady to train for {NUM_EPOCHS} epochs on {len(train_data):,} examples")
print(f"Estimated time: {'2-3 hours' if LITE_MODE else '8-12 hours'}")

## Train! 🚀

In [ ]:
print("Starting training...")
print("This will take a while. Go get coffee! ☕")
print("="*60)

# Train
trainer.train()

print("="*60)
print("✅ Training complete!")

## Save Model

In [ ]:
# Save locally
print(f"Saving model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Model saved locally")

## Push to HuggingFace Hub

In [ ]:
# Push to Hub
print(f"Pushing to HuggingFace Hub: {HUB_MODEL_ID}")
model.push_to_hub(HUB_MODEL_ID, use_auth_token=True)
tokenizer.push_to_hub(HUB_MODEL_ID, use_auth_token=True)

print("✅ Model pushed to Hub")
print(f"\n🔗 View at: https://huggingface.co/{HUB_MODEL_ID}")

## Summary

✅ Fine-tuning complete!

**What we did:**
1. Loaded Llama 3.2 in 4-bit quantization
2. Added LoRA adapters (only 2% trainable parameters)
3. Trained with SFTTrainer on prompt-completion pairs
4. Saved adapters locally and to HuggingFace Hub

**Expected results:**
- Lite mode: ~$65 error
- Full mode: ~$40 error (beats GPT-5.1!)

**Next step:** `04_evaluation.ipynb` - Evaluate the fine-tuned model!